# FLEO — Ablation on subspace width d (Reviewer 1)

Trains YOLOv8n-FLEO on RAF-DB for **d = 4, 8, 16, 32** to justify the paper's d=8.
Each run saves to `runs/abl_d<d>/`. After the runs, the last cell builds a summary table.

Kernel: pick the same Python you used for training (torch CUDA + ultralytics).
Tuned for a 6 GB GPU (RTX 2060): imgsz 128, batch 16, 40 epochs. ~2-3 h total.

## 1. Go to repo root + check GPU

In [ ]:
import os
if not os.path.isdir('scripts'):
    os.chdir('..')
print('cwd:', os.getcwd())
assert os.path.isdir('scripts') and os.path.isdir('fleo'), 'Open from inside the FLEO repo (main branch).'
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'No GPU visible to torch.'

## 2. Fix the RAF-DB dataset path

In [ ]:
import re, pathlib
p = pathlib.Path('datasets/rafdb/data.yaml')
assert p.exists(), 'datasets/rafdb/data.yaml not found'
local = pathlib.Path('datasets/rafdb').resolve().as_posix()
t = re.sub(r'^path:.*', f'path: {local}', p.read_text(), flags=re.M)
p.write_text(t)
print(t.splitlines()[0])

## 3. Run the ablation (d = 4, 8, 16, 32)

In [ ]:
import subprocess, sys
for d in [4, 8, 16, 32]:
    print(f'\n==================  d = {d}  ==================', flush=True)
    r = subprocess.run([
        sys.executable, '-m', 'scripts.train',
        '--data', 'datasets/rafdb/data.yaml', '--variant', 'fleo',
        '--cfg', 'yolov8n.yaml', '--pretrained', 'yolov8n.pt',
        '--d', str(d),
        '--epochs', '40', '--imgsz', '128', '--batch', '16',
        '--device', '0', '--seeds', '0', '--workers', '2',
        '--project', f'runs/abl_d{d}',
    ])
    print(f'd={d} finished, exit code {r.returncode}', flush=True)

## 4. Collect the results into one table

In [ ]:
import pandas as pd, glob
rows = []
for d in [4, 8, 16, 32]:
    csvs = glob.glob(f'runs/abl_d{d}/**/results.csv', recursive=True)
    if not csvs:
        rows.append({'d': d, 'channels(7d)': 7*d, 'note': 'no results.csv'}); continue
    df = pd.read_csv(sorted(csvs)[0])
    df.columns = [c.strip() for c in df.columns]
    best = df.loc[df['metrics/mAP50(B)'].idxmax()]
    rows.append({
        'd': d, 'channels(7d)': 7*d,
        'mAP50': round(float(best['metrics/mAP50(B)']), 4),
        'mAP50-95': round(float(best['metrics/mAP50-95(B)']), 4),
        'precision': round(float(best['metrics/precision(B)']), 4),
        'recall': round(float(best['metrics/recall(B)']), 4),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))
table.to_csv('runs/ablation_d_summary.csv', index=False)
print('\nsaved -> runs/ablation_d_summary.csv')